# Explore GSE176021 CD8 T-cell Annotations (Caushi et al. 2021)

**Dataset**: Caushi et al. 2021 — CD8 T cells from NSCLC patients (anti-PD1 therapy).  
**Files**:
- `data/raw_downloads/GSE176021_CD8_annotations.rds` — original GEO deposit (7 cols)  
- `data/caushi2021_cd8_annotated.rds` — with `is_tumor_reactive` flag added by pipeline  
- `data/caushi2021_ranked_clonotypes.csv` — per-patient clonotype summary  

**Key question**: Are there tumor-reactive clonotypes, and which CD8 subtypes carry them?


In [ ]:
suppressPackageStartupMessages({
  library(ggplot2)
  library(dplyr)
  library(tidyr)
})

# Parameters
rds_raw       <- "data/raw_downloads/GSE176021_CD8_annotations.rds"
rds_annotated <- "data/caushi2021_cd8_annotated.rds"
clonotype_csv <- "data/caushi2021_ranked_clonotypes.csv"

options(repr.plot.width = 10, repr.plot.height = 4.5)


## 1. Load & Overview

In [ ]:
raw <- readRDS(rds_raw)
ann <- readRDS(rds_annotated)
clo <- read.csv(clonotype_csv, stringsAsFactors = FALSE)

cat("=== Raw RDS ===\n")
cat("Cells:", nrow(raw), "  Columns:", ncol(raw), "\n")
cat("Columns:", paste(colnames(raw), collapse = ", "), "\n\n")

cat("=== Annotated RDS (with is_tumor_reactive) ===\n")
cat("Cells:", nrow(ann), "  Columns:", ncol(ann), "\n")
cat("Columns:", paste(colnames(ann), collapse = ", "), "\n\n")

cat("=== Clonotype table ===\n")
print(clo)


## 2. Cell-type Distribution

In [ ]:
ct_counts <- ann %>%
  count(CellType, sort = TRUE) %>%
  mutate(pct = round(100 * n / sum(n), 1))

print(ct_counts)

p <- ggplot(ct_counts, aes(x = reorder(CellType, n), y = n, fill = CellType)) +
  geom_col(show.legend = FALSE) +
  coord_flip() +
  geom_text(aes(label = paste0(pct, "%")), hjust = -0.1, size = 3) +
  scale_y_continuous(expand = expansion(mult = c(0, 0.15))) +
  labs(title = "CD8 T-cell subtypes — Caushi 2021",
       x = NULL, y = "Cells") +
  theme_classic(base_size = 11)
print(p)


## 3. UMAP — Cell Types & Patients

In [ ]:
options(repr.plot.width = 13, repr.plot.height = 5)

p1 <- ggplot(ann, aes(UMAP_1, UMAP_2, colour = CellType)) +
  geom_point(size = 0.15, alpha = 0.4) +
  theme_void(base_size = 10) +
  guides(colour = guide_legend(override.aes = list(size = 3, alpha = 1))) +
  labs(title = "UMAP — Cell Type", colour = NULL)

p2 <- ggplot(ann, aes(UMAP_1, UMAP_2, colour = imid)) +
  geom_point(size = 0.15, alpha = 0.3) +
  theme_void(base_size = 10) +
  guides(colour = guide_legend(override.aes = list(size = 3, alpha = 1), ncol = 2)) +
  labs(title = "UMAP — Patient (imid)", colour = NULL)

gridExtra::grid.arrange(p1, p2, ncol = 2)


## 4. Tumor-reactive Cells

In [ ]:
options(repr.plot.width = 13, repr.plot.height = 5)

cat("=== Tumor-reactive summary ===\n")
print(table(ann$is_tumor_reactive))
cat("Reactive fraction:", round(mean(ann$is_tumor_reactive) * 100, 1), "%\n\n")

# Per cell type reactive fraction
reactive_by_ct <- ann %>%
  group_by(CellType) %>%
  summarise(n_total    = n(),
            n_reactive = sum(is_tumor_reactive),
            pct_reactive = round(100 * mean(is_tumor_reactive), 1)) %>%
  arrange(desc(pct_reactive))
print(reactive_by_ct)


In [ ]:
options(repr.plot.width = 10, repr.plot.height = 4)

p_bar <- ggplot(reactive_by_ct, aes(x = reorder(CellType, pct_reactive), y = pct_reactive)) +
  geom_col(fill = "#e63946", alpha = 0.85) +
  coord_flip() +
  geom_text(aes(label = paste0(pct_reactive, "%")), hjust = -0.1, size = 3) +
  scale_y_continuous(limits = c(0, 105)) +
  labs(title = "% Tumor-reactive cells per CD8 subtype",
       x = NULL, y = "% tumor-reactive") +
  theme_classic(base_size = 11)
print(p_bar)


In [ ]:
options(repr.plot.width = 7, repr.plot.height = 4.5)

p_umap <- ggplot(ann %>% arrange(is_tumor_reactive),
                 aes(UMAP_1, UMAP_2,
                     colour = is_tumor_reactive,
                     alpha  = is_tumor_reactive)) +
  geom_point(size = 0.2) +
  scale_colour_manual(values = c("FALSE" = "grey80", "TRUE" = "#e63946"),
                      labels = c("Non-reactive", "Tumor-reactive")) +
  scale_alpha_manual(values = c("FALSE" = 0.2, "TRUE" = 0.6), guide = "none") +
  theme_void(base_size = 10) +
  guides(colour = guide_legend(override.aes = list(size = 3, alpha = 1))) +
  labs(title = "UMAP — Tumor-reactive cells highlighted", colour = NULL)
print(p_umap)


## 5. Per-patient Clonotype Summary

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 4)

# Patient-level counts
pat_summary <- ann %>%
  group_by(imid) %>%
  summarise(n_cells     = n(),
            n_reactive  = sum(is_tumor_reactive),
            pct_reactive = round(100 * mean(is_tumor_reactive), 1)) %>%
  arrange(desc(pct_reactive))

cat("=== Per-patient reactive fraction ===\n")
print(pat_summary)

p_pat <- ggplot(pat_summary, aes(x = reorder(imid, pct_reactive), y = pct_reactive)) +
  geom_col(fill = "#457b9d", alpha = 0.85) +
  coord_flip() +
  geom_text(aes(label = paste0(pct_reactive, "%")), hjust = -0.1, size = 3) +
  scale_y_continuous(limits = c(0, 105)) +
  labs(title = "% Tumor-reactive cells per patient",
       x = "Patient (imid)", y = "% tumor-reactive") +
  theme_classic(base_size = 11)
print(p_pat)


## 6. Ranked Clonotype Table

In [ ]:
cat("=== caushi2021_ranked_clonotypes.csv ===\n")
print(clo)

cat("\n=== Summary ===\n")
cat("Total clonotypes:", nrow(clo), "\n")
cat("Tumor-reactive:", sum(clo$is_tumor_reactive), "\n")
cat("Total reactive cells:", sum(clo$n_reactive_cells), "\n")
cat("Median clone size:", median(clo$clone_size), "\n")
cat("Mean reactive fraction per clone:", round(mean(clo$n_reactive_cells / clo$clone_size) * 100, 1), "%\n")


In [ ]:
options(repr.plot.width = 10, repr.plot.height = 4.5)

p_clone <- ggplot(clo, aes(x = reorder(patient, -clone_size),
                            y = clone_size, fill = is_tumor_reactive)) +
  geom_col(position = "dodge", alpha = 0.85) +
  scale_fill_manual(values = c("TRUE" = "#e63946", "FALSE" = "grey60"),
                    labels = c("Tumor-reactive", "Non-reactive")) +
  labs(title = "Clone size per patient — Caushi 2021",
       x = "Patient (imid)", y = "Clone size (cells)", fill = NULL) +
  theme_classic(base_size = 11) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1))
print(p_clone)


## 7. Key Findings Summary

In [ ]:
cat("╔══════════════════════════════════════════════╗\n")
cat("║  GSE176021 CD8 T-cell exploration summary   ║\n")
cat("╚══════════════════════════════════════════════╝\n\n")

cat("Dataset: Caushi et al. 2021 (NSCLC, anti-PD1)\n")
cat("  Total cells :", nrow(ann), "\n")
cat("  Patients     :", length(unique(ann$imid)), "\n")
cat("  Cell types   :", length(unique(ann$CellType)), "\n\n")

cat("Clonotypes:\n")
cat("  Unique patient clones  :", nrow(clo), "\n")
cat("  Tumor-reactive clones  :", sum(clo$is_tumor_reactive), "\n")
cat("  Reactive cells total   :", sum(clo$n_reactive_cells), "\n\n")

cat("Top reactive cell types:\n")
print(head(reactive_by_ct[, c("CellType","pct_reactive")], 5))

cat("\nNote: 'is_tumor_reactive' was determined by the Caushi 2021 authors via\n")
cat("neoantigen-stimulation assay + paired TCR-seq. One dominant reactive clonotype\n")
cat("per patient was used (patient imid = clonotype_id).\n")
